# Electronic Structure in Second Quantization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PhilipVinc/ComputationalQuantumPhysics/blob/main/Notebooks/6-Elecs/1-ED.ipynb)

Real molecules are governed by the **Born-Oppenheimer Hamiltonian** (atomic units, nuclei fixed):

$$H = -\frac{1}{2}\sum_i \nabla_i^2 - \sum_{i,A}\frac{Z_A}{|\mathbf{r}_i - \mathbf{R}_A|} + \sum_{i<j}\frac{1}{|\mathbf{r}_i - \mathbf{r}_j|} + E_{\rm nuc}$$

The two-body electron-electron repulsion prevents a simple product-state solution and makes the problem exponentially hard. 
We will use [PySCF](https://pyscf.org/) to:
1. Solve this problem exactly with **Full Configuration Interaction** (FCI) and approximately with **Restricted Hartree-Fock** (RHF).
2. Study how the choice of **basis set** affects accuracy.
3. Extract the matrix elements that appear in the **second-quantized** form of $H$.

In [14]:
# !pip install pyscf  # uncomment if needed

import numpy as np
import matplotlib.pyplot as plt
from pyscf import gto, scf, fci, ao2mo, ci

## Basis Sets

Each molecular orbital is expanded in a set of atom-centered **Gaussian-type functions** $g(\mathbf{r}) \propto e^{-\alpha r^2}$, parametrised by an exponent $\alpha$. In practice one uses two layers:

- **Primitives**: the raw Gaussians $e^{-\alpha_k r^2}$, each with its own exponent.
- **Contracted function**: a *fixed* linear combination of primitives, $\phi(\mathbf{r}) = \sum_k c_k\, e^{-\alpha_k r^2}$, where the coefficients $c_k$ are determined once (from atomic calculations) and never changed during the molecular calculation. The whole combination counts as **one** basis function — one variational degree of freedom.

So `[1s] from (3s)` means: 3 primitive Gaussians are combined with fixed coefficients into a single contracted 1s function. You pay the integral cost of 3 Gaussians but the wavefunction has only 1 free parameter for that shell.

For the **hydrogen atom**, the four bases we use are:

| Basis | Contracted shells on H | What's in them |
|---|---|---|
| `sto-3g` | `[1s]` | 1 contracted s (3 primitives) — one fixed shape for the 1s |
| `6-31g` | `[2s]` | 2 contracted s: a tight one (3 primitives) + a diffuse one (1 primitive, uncontracted) |
| `cc-pvdz` | `[2s, 1p]` | Same 2s split + one set of p functions (px, py, pz) for polarization |
| `cc-pvtz` | `[3s, 2p, 1d]` | Three s + two p + one d — more radial and angular flexibility |

The conceptual steps are:
- `sto-3g` → `6-31g`: split the 1s into a tight and a diffuse part (better radial flexibility).
- `6-31g` → `cc-pvdz`: add **p** functions on H — these capture how the spherical 1s distorts when forming a bond.
- `cc-pvdz` → `cc-pvtz`: add a third s, a second p, and **d** functions for higher-accuracy correlation.

FCI cost grows as $\binom{N_{\rm MO}}{N_\alpha}^2$, so basis choice is a direct accuracy-vs-cost tradeoff.

## Defining a Molecule and Running Solvers

**RHF (Restricted Hartree-Fock)**: variational mean-field — the ground state is approximated as a single Slater determinant. Each electron moves in the average field of the others. Correlations are completely neglected.

**FCI (Full Configuration Interaction)**: exact diagonalization within the chosen basis. The wavefunction is expanded in *all* possible Slater determinants:
$$|\Psi\rangle = \sum_{I} c_I |D_I\rangle$$
This is the exact answer within the basis, at exponential cost.

In [15]:
# Build an H2 molecule at its experimental equilibrium geometry
mol = gto.Mole()
mol.atom = (
    """
    H 0 0 0; 
    H 0 0 0.74
    """
    )  # two H atoms separated by 0.74 Å along z
mol.basis = 'sto-3g'               # pick the basis
mol.unit = 'Angstrom'
mol.verbose = 0                    # suppress PySCF output
mol.build()                        # compute integrals

print(f"Electrons:              {mol.nelectron}")
print(f"AO basis functions:     {mol.nao}")
print(f"Nuclear repulsion:      {mol.energy_nuc():.6f} Ha")

Electrons:              2
AO basis functions:     2
Nuclear repulsion:      0.715104 Ha


In [16]:
# --- Mean-field solution: Restricted Hartree-Fock ---
mf = scf.RHF(mol)   # create the RHF solver bound to this molecule
mf.verbose = 0
mf.kernel()         # run the SCF iterations until convergence

print(f"RHF energy : {mf.e_tot:.6f} Ha")

# --- Exact solution: Full Configuration Interaction ---
cisolver = fci.FCI(mf)   # FCI solver built on top of the RHF MOs
cisolver.verbose = 0
e_fci, ci_vec = cisolver.kernel()   # diagonalise H in the full determinant space

print(f"FCI energy : {e_fci:.6f} Ha")
print(f"Correlation energy (FCI - RHF): {e_fci - mf.e_tot:.6f} Ha")

RHF energy : -1.116759 Ha
FCI energy : -1.137284 Ha
Correlation energy (FCI - RHF): -0.020525 Ha


## H₂ Dissociation Curve

We compute the total energy as a function of bond length $R$ for three bases. Two key quantities to extract:

- **Equilibrium distance** $R_{\rm eq}$: the $R$ that minimizes $E(R)$.
- **Binding energy** $E_{\rm bind} = E(R \to \infty) - E(R_{\rm eq}) > 0$.

Experimental values: $R_{\rm eq} = 0.741$ Å, $E_{\rm bind} = 0.1745$ Ha (4.75 eV).

Note that RHF *fails* at large $R$: a single determinant cannot correctly describe bond breaking, where the true state approaches $\frac{1}{\sqrt{2}}(|\uparrow\rangle_A|\downarrow\rangle_B + |\downarrow\rangle_A|\uparrow\rangle_B)$.

### Task

0. Compute the $E(R)$ for various values of R in a reasonable interval for different choices of bases. Compute the energy both using FCI (Exact diagonalisation) and RHF (Mean field calculation).

1. Plot the FCI dissociation curves for all three bases on the same figure. Add the RHF curve for `sto-3g`. What happens to RHF at large $R$?

2. For each basis extract $R_{\rm eq}$ and $E_{\rm bind}$ from the FCI curve. Compare to experiment.

3. Make a summary plot with $R_{\rm eq}$ and $E_{\rm bind}$ as a function of basis, including the experimental reference as a horizontal line.

In [ ]:
# --- 0. Loop through different bases and R molecules, and for each compute 
#        the RHF and FCI energies. Store the results in a dictionary for later use.
# ...

# --- 1. Plot the FCI dissociation curves for all bases on the same figure.
#        Add the RHF curve for sto-3g. Comment on what happens at large R.
# ...

# --- 2. Print a table of R_eq and E_bind for each basis and compare to experiment.
#        Experimental values: R_eq = 0.741 Å, E_bind = 0.1745 Ha (4.75 eV).
# ...

# --- 3. Make a bar chart comparing R_eq and E_bind across bases.
#        Add a horizontal line for the experimental value.
# ...

---
## Assignment: N₂ Dissociation

N₂ has a triple bond and is a standard benchmark in quantum chemistry. Repeat the dissociation-curve study you did for H₂.

A `make_n2` helper and a check of the FCI space sizes are given below. Note that `cc-pvdz` is far too large for FCI — for that basis restrict yourself to RHF.

Experimental reference values: $R_{\rm eq} = 1.098$ Å, $E_{\rm bind} \approx 0.364$ Ha (9.9 eV).

1. Compute FCI and RHF dissociation curves for `sto-3g`. Add the RHF curve for `cc-pvdz`. Plot everything on one figure.
2. Extract $R_{\rm eq}$ and $E_{\rm bind}$ for each method/basis and compare to experiment.
3. At $R = 3.0$ Å look at the dominant CI coefficients. How many determinants contribute significantly? How does this compare to H₂ at dissociation?

In [32]:
def make_n2(R, basis='sto-3g'):
    mol = gto.Mole()
    mol.atom = f'N 0 0 0; N 0 0 {R}'
    mol.basis = basis
    mol.unit = 'Angstrom'
    mol.verbose = 0
    mol.build()
    return mol

# Check the FCI space sizes
from math import comb
for basis in ['sto-3g', 'cc-pvdz']:
    mol = make_n2(1.098, basis)
    na, nb = mol.nelec
    nao = mol.nao
    fci_dim = comb(nao, na) * comb(nao, nb)
    print(f"{basis:<10}: N_AO={nao}, nelec=({na},{nb}), FCI dim = {fci_dim:,}")

sto-3g    : N_AO=10, nelec=(7,7), FCI dim = 14,400
cc-pvdz   : N_AO=28, nelec=(7,7), FCI dim = 1,401,950,721,600


In [ ]:
R_n2 = np.linspace(0.9, 3.5, 30)

# --- your code here ---

## For the curious: The Hamiltonian in Second Quantization and constructing it from the molecule object

(You can safely skip this section)

After choosing a basis of $N$ molecular orbitals $\{\phi_p\}$, the Hamiltonian takes the form

$$H = \sum_{pq} h_{pq}\, \hat{a}_p^\dagger \hat{a}_q + \frac{1}{2}\sum_{pqrs} g_{pqrs}\, \hat{a}_p^\dagger \hat{a}_r^\dagger \hat{a}_s \hat{a}_q + E_{\rm nuc}$$

where
$$h_{pq} = \langle \phi_p | -\tfrac{1}{2}\nabla^2 - \sum_A \tfrac{Z_A}{|r-R_A|} | \phi_q \rangle, \qquad
g_{pqrs} = \int \frac{\phi_p^*(r)\,\phi_q(r)\,\phi_r^*(r')\,\phi_s(r')}{|r-r'|}\,dr\,dr'$$

PySCF computes these integrals and stores them in the atomic orbital (AO) basis; we transform them to the MO basis using the coefficient matrix $C$ (columns = MO coefficients in the AO basis):
$$h^{\rm MO}_{pq} = \sum_{\mu\nu} C_{\mu p}\, h^{\rm AO}_{\mu\nu}\, C_{\nu q}$$

In [ ]:
def make_h2(R, basis='sto-3g'):
    """Helper function to build an H2 molecule at distance R with a given basis."""
    mol = gto.Mole()
    mol.atom = (
        f"""
        H 0 0 0; 
        H 0 0 {R}
        """
    )
    mol.basis = basis
    mol.unit = 'Angstrom'
    mol.verbose = 0
    mol.build()
    return mol

def run_rhf(mol):
    """Run RHF and return the total energy."""
    mf = scf.RHF(mol)
    mf.verbose = 0
    mf.kernel()
    return mf

def run_fci(mf):
    """Run FCI on top of the given RHF solution and return the total energy."""
    cisolver = fci.FCI(mf)
    cisolver.verbose = 0
    e_fci, ci_vec = cisolver.kernel()
    return e_fci, ci_vec, cisolver

In [ ]:
# Extract matrix elements for H2 at equilibrium in sto-3g
mol = make_h2(0.74, basis='sto-3g')
mf  = run_rhf(mol)
C   = mf.mo_coeff          # AO -> MO coefficient matrix, shape (N_AO, N_MO)

# -- One-electron integrals in AO basis --
T_ao = mol.intor('int1e_kin')   # kinetic energy
V_ao = mol.intor('int1e_nuc')   # nuclear attraction

# -- Transform to MO basis --
T_mo = C.T @ T_ao @ C
V_mo = C.T @ V_ao @ C
h_mo = T_mo + V_mo             # = h_pq  (one-body part of H)

# -- Two-electron integrals in MO basis: g[p,q,r,s] = (pq|rs) --
nmo = C.shape[1]
eri_mo = ao2mo.restore(1, ao2mo.kernel(mol, C), nmo)  # shape (nmo,nmo,nmo,nmo)

print(f"C  shape: {C.shape}      (N_AO x N_MO)")
print(f"h  shape: {h_mo.shape}    (MO basis)")
print(f"ERI shape: {eri_mo.shape}  (MO basis)")
print()
print("h_pq (kinetic + nuclear-attraction):")
print(np.round(h_mo, 4))
print()
print("E_nuc =", round(mol.energy_nuc(), 6), "Ha")

C  shape: (2, 2)      (N_AO x N_MO)
h  shape: (2, 2)    (MO basis)
ERI shape: (2, 2, 2, 2)  (MO basis)

h_pq (kinetic + nuclear-attraction):
[[-1.2533 -0.    ]
 [-0.     -0.4751]]

E_nuc = 0.715104 Ha


In [ ]:
# Reconstruct the total energy from integrals + FCI coefficients as a sanity check

e_fci, ci_vec, cisolver = run_fci(mf)

norb  = nmo
na, nb = mol.nelec   # alpha, beta electron counts

# cisolver.energy(h1e, eri, fcivec, norb, nelec)
e_check = cisolver.energy(h_mo, eri_mo, ci_vec, norb, (na, nb))
print(f"FCI energy (direct):      {e_fci:.8f} Ha")
print(f"FCI energy (from h, ERI): {e_check + mol.energy_nuc():.8f} Ha")

FCI energy (direct):      -1.13728383 Ha
FCI energy (from h, ERI): -1.13728383 Ha


---
## Reading the FCI Wavefunction

The FCI ground state is a superposition of all Slater determinants:
$$|\Psi_0\rangle = \sum_{I_\alpha, I_\beta} c_{I_\alpha I_\beta}\, |\alpha{:}I_\alpha,\ \beta{:}I_\beta\rangle$$

PySCF stores the coefficients in a 2D array `ci_vec[I_α, I_β]` where each index encodes an occupation pattern as a bitstring: if bit $k$ of $I_\alpha$ is set, orbital $k$ is occupied by an $\alpha$-electron.

**H₂ in STO-3G** is the simplest case: 1 $\alpha$- and 1 $\beta$-electron in 2 spatial orbitals (bonding $\sigma$ and antibonding $\sigma^*$) → a 2×2 matrix.

| Row/col index | Occupied orbital | Physical picture |
|---|---|---|
| 0 | $\sigma$ (orbital 0) | bonding — the Hartree-Fock determinant |
| 1 | $\sigma^*$ (orbital 1) | antibonding |

So `ci_vec[0,0]` is the amplitude of the Hartree-Fock state (both electrons in the bonding orbital), and `ci_vec[1,1]` is the amplitude of its doubly-excited counterpart.

In [ ]:
# Look at how the CI matrix evolves as the bond stretches
for R, label in [(0.74, 'equilibrium'), (1.5, 'stretched'), (4.5, 'dissociation')]:
    mol = make_h2(R)
    mf  = run_rhf(mol)
    e, c, _ = run_fci(mf)
    print(f"R = {R:.2f} Å  ({label})")
    print(f"  ci_vec:\n  {np.round(c, 4)}")
    print(f"  weight of HF det  |c[0,0]|² = {c[0,0]**2:.4f}")
    print(f"  weight of exc det |c[1,1]|² = {c[1,1]**2:.4f}")
    print()

At equilibrium the HF determinant carries ~98% of the weight — a single Slater determinant is an excellent approximation. Near dissociation the two determinants become equally weighted (~50% each): the system is **strongly correlated** and any method built on a dominant single determinant (RHF, MP2, coupled-cluster) breaks down.

The weight of the HF determinant,
$$w_0 = |c_{00}|^2,$$
is the simplest diagnostic for this. When $w_0 \lesssim 0.9$ the system is called *multi-reference* and single-reference methods are unreliable.

### Exercise

1. For **H₂ in `sto-3g`**, compute $w_0(R)$ across a grid of bond lengths and plot it as a function of $R$. At roughly what distance does the system become strongly multi-reference?

2. For **N₂ in `sto-3g`**, the `ci_vec` is 120×120. Use `fci.addons.large_ci(ci_vec, norb, nelec, tol=0.05)` to list the dominant determinants at $R = 1.1$ Å and $R = 3.0$ Å. How many determinants contribute significantly compared to H₂? What does this say about the difficulty of describing the dissociation of a triple bond?

In [ ]:
# --- Exercise 1: w0(R) for H2 ---
# ...

# --- Exercise 2: dominant determinants in N2 ---
# Hint:
#   mol = make_n2(R, 'sto-3g')
#   mf  = run_rhf(mol); run_fci(mf) → e, ci_vec, cisolver
#   norb, nelec = mol.nao, mol.nelec
#   large_dets = fci.addons.large_ci(ci_vec, norb, nelec, tol=0.05)
#   # each entry is (coeff, occ_alpha, occ_beta)
# ...